# Everything You Get From Flyte Without a Cluster

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/starter-examples/flyte-local-dev/tutorial_flyte_local_dev.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

`pip install flyte` gives you a powerful local development toolkit — caching, reports, tracing, and run history — all backed by a local SQLite database.

**What you'll learn:**
- Cache task outputs to skip recomputation (save time and API costs)
- Generate HTML reports from tasks
- Use `@flyte.trace` for sub-task observability
- Browse run history with the TUI (terminal only)
- Serve tasks as local API endpoints

---

## Setup

You can clone repo: `git clone https://github.com/unionai/workshops`

If you're on [GitHub CodeSpaces](https://codespaces.new/unionai/workshops) you can copy below into your terminal from the root directory `/workshops`

Note that shouldn't need `keyrings.alt` on local environments that already have keyrings set (Mac, windows, etc)

If you already have UV installed skip the first curl command | `.venv\Scripts\activate` to activate for windows


```
curl -LsSf https://astral.sh/uv/install.sh | sh
uv venv .venv --python 3.11
source .venv/bin/activate
cd tutorials/starter-examples/flyte-local-dev
uv pip install -r requirements.txt
uv pip install keyrings.alt 
```

In [2]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/starter-examples/flyte-local-dev/
    !pip install -r requirements.txt

from utils.file_viewer import view_file

## Dependencies

In [3]:
view_file("requirements.txt")

## Set your API Key

The agent examples use OpenAI. Set it as an environment variable, in a `.env`, or enter it below:

In [4]:
import os
from getpass import getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')

---

## 1. Caching: Skip Work You've Already Done

An LLM call costs money and takes seconds. A dataset download takes minutes. If you're iterating on downstream logic, you shouldn't have to redo all of that every run.

Add `cache="auto"` to any task — Flyte stores outputs in local SQLite keyed on task name + inputs. Same inputs = instant result.

### Agent Example: Cache Expensive LLM Calls

In [5]:
view_file("cached_agent.py")

In [6]:
# First run — calls OpenAI
!flyte run --local cached_agent.py agent --request "What is 12 * 7 plus 3?"

Running agent for: What is 12 * 7 plus 3?
⠸ Launching local execution...19:29:20.478716 ERROR    taskrunner.py:104 -                                    
                         [6be36878-82aa-4b43-8753-8b8d5137a9a9][4htech7zex5bmv3e
                         9n99dr00i]  Task failed with error: Error code: 401 -  
                         {'error': {'message': "You didn't provide an API key.  
                         You need to provide your API key in an Authorization   
                         header using Bearer auth (i.e. Authorization: Bearer   
                         YOUR_KEY), or as the password field (with blank        
                         username) if you're accessing the API from your browser
                         and are prompted for a username and password. You can  
                         obtain an API key from                                 
                         https://platform.openai.com/account/api-keys.", 'type':
                         'invalid_req

In [7]:
# Second run — cache hit, returns instantly (no API call)
!flyte run --local cached_agent.py agent --request "What is 12 * 7 plus 3?"

Running agent for: What is 12 * 7 plus 3?
⠸ Launching local execution...19:29:23.165590 ERROR    taskrunner.py:104 -                                    
                         [9eb1dddd-b282-43cb-8381-da39beae75c5][3unti3tjc8dzfd91
                         83dg0lrj6]  Task failed with error: Error code: 401 -  
                         {'error': {'message': "You didn't provide an API key.  
                         You need to provide your API key in an Authorization   
                         header using Bearer auth (i.e. Authorization: Bearer   
                         YOUR_KEY), or as the password field (with blank        
                         username) if you're accessing the API from your browser
                         and are prompted for a username and password. You can  
                         obtain an API key from                                 
                         https://platform.openai.com/account/api-keys.", 'type':
                         'invalid_req

### ML Example: Cache Dataset Loading and Preprocessing

In [ ]:
view_file("cached_ml_pipeline.py")

In [ ]:
# First run — loads data, splits, trains, evaluates
!flyte run --local cached_ml_pipeline.py pipeline --n_neighbors 3

In [ ]:
# load_data and split_data are cache hits, only train + evaluate re-run
!flyte run --local cached_ml_pipeline.py pipeline --n_neighbors 5

In [ ]:
# Same args as first run — everything is cached
!flyte run --local cached_ml_pipeline.py pipeline --n_neighbors 3

### Lightweight Experiment Tracking

Each run with different hyperparameters is a separate entry in the local database. You can browse and compare them in the TUI:

```bash
flyte start tui
```

---

## 2. Reports: Generate HTML Dashboards From Your Tasks

A confusion matrix in your terminal isn't very useful. Neither is a wall of JSON from an agent trace. Add `report=True` to any task and use `flyte.report` to generate HTML — charts, tables, images — saved alongside the output.

### Agent Example: Log the Reasoning Trace

In [ ]:
view_file("agent_with_report.py")

In [ ]:
!flyte run --local agent_with_report.py agent --request "What is 12 * 7 plus 3?"

Open the `report.html` from the output path in your browser to see the full agent reasoning trace.

The ML pipeline (`cached_ml_pipeline.py`) also generates a report with a confusion matrix and classification report from the `evaluate` task.

---

## 3. TUI + Tracing: Live Dashboard in Your Terminal

The TUI gives you a real-time split-screen view of any local run. Add `@flyte.trace` to functions for sub-task visibility — traced functions show up as child nodes in the action tree.

**Run these from your terminal** (won't render in the notebook):

```bash
# Watch the agent run step by step
flyte run --local --tui cached_agent.py agent --request "What is 12 * 7 plus 3?"

# Watch the ML pipeline — see cache hits on repeated runs
flyte run --local --tui cached_ml_pipeline.py pipeline --n_neighbors 3

# Browse all past runs
flyte start tui
```

**What you see:**
- Task tree with live status: `●` running, `✓` succeeded, `✗` failed
- Cache indicators: `$` cache hit, `~` cache miss
- `@flyte.trace` functions as child nodes under their parent task
- Task inputs, outputs, duration, errors
- Real-time logs

**Keyboard shortcuts:** `q` quit, `d` details, `l` logs

---

## 4. Local Serving: Run Tasks as API Endpoints

Flyte can serve a FastAPI app locally that calls your tasks — no cluster required. Train a model, then serve predictions:

In [ ]:
view_file("serve_model.py")

**Run from your terminal:**

```bash
python serve_model.py
```

Then hit the endpoint:

```bash
curl "http://localhost:8080/predict?sepal_length=5.1&sepal_width=3.5&petal_length=1.4&petal_width=0.2"
```

The key API: `flyte.with_servecontext(mode="local").serve(app_env)` — runs in-process, no container build. Switch to `mode="remote"` to deploy to your cluster.

---

## Summary

No cluster. No Docker. No config. Just `pip install flyte` and you get:

| Feature | What it does |
|---------|-------------|
| **TUI + Tracing** | Live terminal dashboard, sub-task observability, run history browser |
| **Caching** | Skip recomputation on repeated inputs |
| **Reports** | HTML dashboards generated from tasks |
| **Local serving** | Serve tasks as FastAPI endpoints locally |
| **Run history** | All inputs, outputs, timing persisted in SQLite |

And when you're ready to scale, the same code runs on a remote Flyte cluster — just swap `flyte run --local` for `flyte run`.

## Next Steps

- [LangGraph ReAct Agent](../langgraph-react-agent/) — Build an agent and run it remotely
- [Image Classifier](../image-classifier/) — Fine-tune ResNet18 with PyTorch
- [Multi-agent tutorials](../../multi-agent-workflows/) — Complex agent patterns